# NeXo v3.0 · Notebook 03 — OSS Experience Anomaly (Variational Autoencoder)

> **CRISP-DM Phase 4 — Modeling** · Unsupervised anomaly detection on OSS cell KPIs.
> **Input:** `curated/cells.parquet` (per-cell OSS KPIs from nb 01).
> **Output:** `models/oss_vae_v3.pt` + `vae_v3_scaler.joblib` + `vae_v3_feature_names.joblib` + model card.
> **Consumed by:** `services/ai-service/model_cache.py` (mtime hot-reload) → `POST /infer/vae-anomaly`.

## Pipeline
```
cells.parquet → select 9 KPIs → StandardScaler → train VAE on NORMAL cells only
            → reconstruction error per cell → ROC-AUC / PR-AUC → save .pt + scaler
```

## Why VAE over IsolationForest
- **Continuous severity score** (reconstruction MSE) — not just a binary flag.
- **Latent space is interpretable** — anomaly types cluster (UMAP-able).
- **Handles multi-modal traffic** — 2G/3G/4G mixtures the IsolationForest splits poorly.

## Core idea
Train ONLY on normal cells. The VAE learns to reconstruct normal KPI vectors. At inference,
abnormal cells reconstruct poorly → high MSE → high anomaly score. No labels needed at train time.

## Architecture
`Input(9) → Dense(hidden, ReLU) → Dense(mid, ReLU) → Latent(μ,σ) → Decoder(mirror)`
Loss = `MSE(reconstruction) + KL_BETA · KL_divergence`

## What this notebook does NOT do
- Does NOT compute the CEM score (nb 02) or RAT underservice (nb 04).
- Does NOT label anomalies by hand — the weak label `y` is only for *evaluation*, never for training.
- Does NOT deploy — it writes the `.pt` artifact ai-service hot-loads.


## 1 · Imports + GPU detection

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from IPython.display import display

import io, json, os
from pathlib import Path
import boto3, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from botocore.client import Config
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
sns.set_theme(style='whitegrid')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} · device = {DEVICE}')
if DEVICE=='cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')
s3 = boto3.client('s3', endpoint_url=os.environ.get('S3_ENDPOINT','http://localhost:9000'),
                  aws_access_key_id='minio', aws_secret_access_key='minio_pw',
                  config=Config(signature_version='s3v4'))

In [ ]:
# ── Parameters (papermill-overridable) ──────────────────────────────────────
# This cell is tagged `parameters`. The retrain-service (papermill) can override
# any value here without editing the notebook. EVERY magic number lives here.
import os, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
except Exception:
    pass

# Architecture (input dim is data-driven = number of selected KPIs)
VAE_HIDDEN_DIM = 32     # encoder layer 1 width
VAE_MID_DIM    = 16     # encoder layer 2 width
VAE_LATENT_DIM = 8      # latent bottleneck (μ, log_var dimensionality)

# Training
VAE_EPOCHS  = 30        # full passes over normal-cell train set
VAE_BATCH   = 256       # minibatch size
VAE_LR      = 1e-3      # Adam learning rate
VAE_KL_BETA = 0.001     # KL weight (β-VAE). Low β → latent absorbs more signal,
                        #   reconstruction prioritized over a clean Gaussian latent.
VAE_TRAIN_FRAC = 0.80   # fraction of NORMAL cells used for training (rest = val)

# Artifact filenames — IMMUTABLE (services/ai-service/model_cache.py reads these)
VAE_MODEL_FNAME   = 'oss_vae_v3.pt'
VAE_SCALER_FNAME  = 'vae_v3_scaler.joblib'
VAE_FEATS_FNAME   = 'vae_v3_feature_names.joblib'
VAE_CARD_FNAME    = 'oss_vae_v3_model_card.md'

print(f'SEED={SEED} | arch={VAE_HIDDEN_DIM}->{VAE_MID_DIM}->latent({VAE_LATENT_DIM}) | '
      f'epochs={VAE_EPOCHS} batch={VAE_BATCH} lr={VAE_LR} kl_beta={VAE_KL_BETA}')


## 2 · Load curated cells data + build anomaly label

In [ ]:
df = pd.read_parquet(io.BytesIO(s3.get_object(Bucket='curated', Key='cells.parquet')['Body'].read()))
print(f'cells: {len(df):,} × {df.shape[1]}')
integ_c = next((c for c in df.columns if c.lower().startswith('integrity')), None)
cdr_c   = next((c for c in df.columns if 'call drop' in c.lower()), None)
df[integ_c] = pd.to_numeric(df[integ_c], errors='coerce')
df[cdr_c] = pd.to_numeric(df[cdr_c], errors='coerce').fillna(0) if cdr_c else 0
df['anomaly_label'] = ((df[integ_c] < 100) | (df[cdr_c] > 2)).astype(int)
print(df.anomaly_label.value_counts(normalize=True).round(4))

## 3 · Feature selection — 9 OSS KPIs

In [ ]:
def col(df, frag):
    for c in df.columns:
        if frag.lower() in c.lower(): return c
    return None
FEATS = []
for f in ['integrity','call drop','throughput','rsrp','user.avg','user.max',
         'latency_ms_derived','packet_loss_pct_derived','jitter_ms_derived']:
    c = col(df, f)
    if c: FEATS.append(c)
print(f'features ({len(FEATS)}): {FEATS}')
X = df[FEATS].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(np.float32)
y = df['anomaly_label'].values
print(f'X shape: {X.shape}  pos rate: {y.mean():.3f}')

## 4 · Train/val split — NORMAL ONLY in train

VAE never sees anomalies during training. At inference, abnormal cells produce high error.

In [ ]:
scaler = StandardScaler()
Xn = X[y==0]; Xa = X[y==1]
n_tr = int(len(Xn)*VAE_TRAIN_FRAC)
perm = np.random.RandomState(SEED).permutation(len(Xn))
X_tr = scaler.fit_transform(Xn[perm[:n_tr]])
X_vn = scaler.transform(Xn[perm[n_tr:]])
X_va = scaler.transform(Xa)
print(f'train normal: {X_tr.shape}   val normal: {X_vn.shape}   val anom: {X_va.shape}')

## 5 · VAE architecture

Encoder → μ, log_var → reparam → Decoder. Reparameterization trick keeps gradients flowing.

In [ ]:
class VAE(nn.Module):
    def __init__(self, in_dim, hidden=32, mid=16, latent=8):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(in_dim,hidden), nn.ReLU(), nn.Linear(hidden,mid), nn.ReLU())
        self.mu  = nn.Linear(mid, latent)
        self.log = nn.Linear(mid, latent)
        self.dec = nn.Sequential(nn.Linear(latent,mid), nn.ReLU(), nn.Linear(mid,hidden), nn.ReLU(), nn.Linear(hidden,in_dim))
    def reparam(self, mu, log):
        return mu + torch.exp(0.5*log) * torch.randn_like(log)
    def forward(self, x):
        h = self.enc(x); mu, log = self.mu(h), self.log(h)
        return self.dec(self.reparam(mu, log)), mu, log
model = VAE(in_dim=X_tr.shape[1], hidden=VAE_HIDDEN_DIM, mid=VAE_MID_DIM, latent=VAE_LATENT_DIM).to(DEVICE)
print(model)

## 6 · Training loop

Loss = MSE + 0.001 × KL. Low KL beta lets latent absorb signal.

In [ ]:
# hyperparameters come from the parameters cell (papermill-overridable)
EPOCHS, BATCH, LR, KL_BETA = VAE_EPOCHS, VAE_BATCH, VAE_LR, VAE_KL_BETA
loader = DataLoader(TensorDataset(torch.tensor(X_tr)), batch_size=BATCH, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=LR)
model.train(); hist=[]
for ep in range(1, EPOCHS+1):
    tot=0.0
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        rec, mu, log = model(xb)
        mse = ((rec-xb)**2).mean()
        kl = -0.5*(1+log-mu.pow(2)-log.exp()).mean()
        loss = mse + KL_BETA*kl
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item()*len(xb)
    avg = tot/len(X_tr); hist.append(avg)
    if ep%5==0 or ep==1: print(f'epoch {ep:>3d} loss={avg:.5f}')

## 7 · Evaluate — reconstruction error per cell

In [ ]:
def recon(model, X):
    model.eval()
    with torch.no_grad():
        x = torch.tensor(X).to(DEVICE)
        r,_,_ = model(x)
        return ((r-x)**2).mean(dim=1).cpu().numpy()
en = recon(model, X_vn); ea = recon(model, X_va)
yall = np.concatenate([np.zeros(len(en)), np.ones(len(ea))])
eall = np.concatenate([en, ea])
roc = roc_auc_score(yall, eall)
prec, rec, _ = precision_recall_curve(yall, eall)
pr_auc = auc(rec, prec)
print(f'ROC-AUC: {roc:.4f}  PR-AUC: {pr_auc:.4f}')

## 8 · Error distribution plot

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
sns.kdeplot(en, label='normal', ax=ax, color='#10b981', linewidth=2)
sns.kdeplot(ea, label='anomaly', ax=ax, color='#ef4444', linewidth=2)
ax.set_xlabel('recon error (MSE)'); ax.set_title(f'VAE recon error — ROC-AUC={roc:.3f}')
ax.legend(); plt.show()

## 9 · Save model + scaler + push to MinIO

In [ ]:
MODELS = Path('models'); MODELS.mkdir(exist_ok=True)
checkpoint = {
    'model_state_dict': model.state_dict(),
    'input_dim': X_tr.shape[1],
    'latent_dim': VAE_LATENT_DIM,
    'hidden_dim': VAE_HIDDEN_DIM,
    'mid_dim': VAE_MID_DIM,
    'roc_auc': float(roc),
    'features': FEATS,
}
torch.save(checkpoint, MODELS/VAE_MODEL_FNAME)
joblib.dump(scaler, MODELS/VAE_SCALER_FNAME)
joblib.dump(FEATS, MODELS/VAE_FEATS_FNAME)
card = (f'# VAE v3 Model Card\n\n'
        f'**ROC-AUC**: {roc:.4f}\n'
        f'**PR-AUC**: {pr_auc:.4f}\n'
        f'Features: {FEATS}\n')
(MODELS/VAE_CARD_FNAME).write_text(card); print(card)
for fn in [VAE_MODEL_FNAME, VAE_SCALER_FNAME, VAE_FEATS_FNAME, VAE_CARD_FNAME]:
    s3.put_object(Bucket='curated', Key=f'models/{fn}', Body=(MODELS/fn).read_bytes())
    print(f'  ↑ curated/models/{fn}')

---
## Done · register the model

Copy `models/oss_vae_v3.pt` + `vae_v3_scaler.joblib` to `services/ai-service/models/`.
Hot-reload via `POST /models/reload`.